# Candidate Screening Score — QLoRA Fine-Tuning with Qwen

Builds on everything diagnosed in the BERT-oriented LoRA notebook, but
fixed for Qwen's actual architecture from the start rather than patched
error-by-error:

- **QLoRA, not plain LoRA** — the base model loads in 4-bit (NF4)
  quantization, cutting memory substantially versus loading Qwen at full
  precision, before LoRA adapters are attached on top.
- **Correct Qwen module names** — `q_proj`/`k_proj`/`v_proj`/`o_proj`/
  `gate_proj`/`up_proj`/`down_proj` for LoRA's `target_modules`, and
  `score` (not BERT's `classifier`) for `modules_to_save` — Qwen2's
  regression head is genuinely named differently.
- **Qwen's tokenizer has no pad token by default** — set explicitly,
  both on the tokenizer and the model config, before any batched
  operation touches it.
- **Stratified train/val/test split by job title**, so no title is
  over/under-represented across splits by chance.
- **Explicit leakage checks** — no candidate id repeated across splits,
  and `screening_score` never appears as a standalone number inside the
  model's input text (checked with word-boundary matching, not a raw
  substring search, which produces false positives on coincidental
  digits).
- **Every `TrainingArguments` field explained**, not just set.

Setup:
```
pip install -q -U transformers peft accelerate datasets scikit-learn scipy "bitsandbytes>=0.46.1" torchao
```
Restart the runtime after this install, before running anything else — a
few of these libraries do version-sensitive setup at import time.

In [1]:
pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## Getting the dataset — Kaggle private dataset attachment

Attach your dataset via the notebook's right sidebar (**"+ Add Input"**),
then it's available read-only at `/kaggle/input/<slug>/`. Only
`/kaggle/working/` is writable in this session.

In [2]:
import bitsandbytes
print(bitsandbytes.__version__)

0.50.2


In [3]:
INPUT_DIR = "/kaggle/input/datasets/divyeshbhavsar/candidate-extracted"    # <-- update to match your attached dataset
OUTPUT_DIR = "/kaggle/working"

In [4]:
import json
import re
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,EarlyStoppingCallback
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load the data

Robust loading: finds the `screening_score` column even if its exact name
differs slightly, shows raw values before coercing to numeric, and
reports exactly how many rows get dropped and why — rather than silently
coercing and moving on.

In [5]:
DATA_PATH = f"{INPUT_DIR}/candidate_extracted (2).json"   # <-- update to your actual filename

with open(DATA_PATH, "r") as f:
    records = json.load(f)

df = pd.DataFrame(records)
print(f"Loaded {len(df)} raw records")
print(f"Columns: {df.columns.tolist()}")

# Stable unique id -- needed later to prove no candidate leaks across splits.
if "id" not in df.columns:
    df["id"] = df.index

# Find screening_score robustly (handles minor name/casing differences).
score_col = next((c for c in df.columns if c.strip().lower().replace(" ", "_") == "screening_score"), None)
if score_col is None:
    raise KeyError(f"No column matching 'screening_score' found. Columns: {df.columns.tolist()}")
if score_col != "screening_score":
    df = df.rename(columns={score_col: "screening_score"})

print(f"\nscreening_score dtype before coercion: {df['screening_score'].dtype}")
before_n = len(df)
df["screening_score"] = pd.to_numeric(df["screening_score"], errors="coerce")
df = df.dropna(subset=["screening_score"]).reset_index(drop=True)
print(f"Rows before/after dropping unparseable screening_score: {before_n} / {len(df)}")

df.head()

Loaded 1266 raw records
Columns: ['id', 'raw_text', 'screening_score', 'location', 'job_title', 'experience', 'education', 'other_info', '_error']

screening_score dtype before coercion: int64
Rows before/after dropping unparseable screening_score: 1266 / 1266


,id,raw_text,screening_score,location,job_title,experience,education,other_info,_error
0,1,innovative and driven professional seeking a r...,100,United States,data scientist,student/intern-level,None,None,None
1,2,ms applied data science student usc research a...,100,United States,data scientist,student/intern-level,"ms applied data science, usc","research assistant, usc, former data analytics...",None
2,3,computer science student seeking full-time sof...,100,United States,software engineer,student/intern-level,computer science student,"ai, sql, data visualization, python, ssrs",None
3,4,microsoft certified power bi data analyst mba ...,100,United States,data analyst,student/intern-level,"mba, unt business intelligence engineering, ms...","power bi, business analytics, predictive analy...",None
4,5,graduate research assistant at uab masters in ...,100,United States,data scientist,student/intern-level,"masters in data science, uab",None,None


## 2. Build the model input text (leakage-safe by construction)

Only `job_title`, `experience`, `education`, `other_info`, `location` go
into `input_text`. `screening_score` is never referenced here at all —
this is the actual guarantee against leakage; the check in Section 4 is
a verification of this, not the mechanism itself.

In [6]:
def build_input_text(row) -> str:
    parts = []
    for field, label in [
        ("job_title", "Job Title"), ("experience", "Experience"),
        ("education", "Education"), ("other_info", "Other Info"),
        ("location", "Location"),
    ]:
        value = row.get(field)
        if pd.notna(value) and str(value).strip():
            parts.append(f"{label}: {value}")
    return " | ".join(parts)


df["input_text"] = df.apply(build_input_text, axis=1)
df[["input_text", "screening_score"]].head()

,input_text,screening_score
0,Job Title: data scientist | Experience: studen...,100
1,Job Title: data scientist | Experience: studen...,100
2,Job Title: software engineer | Experience: stu...,100
3,Job Title: data analyst | Experience: student/...,100
4,Job Title: data scientist | Experience: studen...,100


## 3. Stratified train/val/test split, by job title

Every job title should appear in train/val/test in roughly the same
proportion it appears in the full dataset, so the model isn't
accidentally trained mostly on one title and validated mostly on another.

Titles with too few rows can't survive two successive stratified splits
reliably (sklearn's `stratify=` needs at least 2 members per class per
split, and rounding can push a small class below that) — those get
merged into one `"Other"` bucket for stratification purposes only; the
real `job_title` value on each row is untouched.

In [7]:
def get_primary_title(job_title) -> str:
    if pd.isna(job_title) or not str(job_title).strip():
        return "Unknown"
    first = re.split(r"[,/]", str(job_title))[0]
    return first.strip().lower()


df["primary_title"] = df["job_title"].apply(get_primary_title)

# Needs to survive train/temp (70/30) then temp/val/test (50/50) --
# rounding can leave a 6-member class with only 1 in temp_df, which
# crashes the second split. 10 gives real margin.
MIN_SAMPLES_PER_CLASS = 10

title_counts = df["primary_title"].value_counts()
rare_titles = title_counts[title_counts < MIN_SAMPLES_PER_CLASS].index
df["strat_title"] = df["primary_title"].where(~df["primary_title"].isin(rare_titles), "Other")

print("Title distribution used for stratification:")
print(df["strat_title"].value_counts())

Title distribution used for stratification:
strat_title
data scientist               409
Other                        317
data analyst                 271
Unknown                       75
software engineer             47
data engineer                 43
machine learning engineer     31
student                       21
software developer            16
researcher                    15
business analyst              11
senior data analyst           10
Name: count, dtype: int64


In [8]:
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

try:
    train_df, temp_df = train_test_split(
        df, test_size=(VAL_FRAC + TEST_FRAC), stratify=df["strat_title"], random_state=SEED,
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
        stratify=temp_df["strat_title"], random_state=SEED,
    )
except ValueError as e:
    raise ValueError(
        f"Stratified split failed: {e}\n\nRaise MIN_SAMPLES_PER_CLASS "
        f"(currently {MIN_SAMPLES_PER_CLASS}) and re-run from the title-bucketing cell."
    ) from e

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

comparison = pd.DataFrame({
    "full": df["strat_title"].value_counts(normalize=True),
    "train": train_df["strat_title"].value_counts(normalize=True),
    "val": val_df["strat_title"].value_counts(normalize=True),
    "test": test_df["strat_title"].value_counts(normalize=True),
}).fillna(0).round(3)
comparison

train: 886  val: 190  test: 190


,full,train,val,test
strat_title,,,,
Other,0.250,0.251,0.253,0.247
Unknown,0.059,0.059,0.063,0.058
business analyst,0.009,0.009,0.005,0.011
data analyst,0.214,0.214,0.211,0.216
data engineer,0.034,0.034,0.032,0.037
data scientist,0.323,0.323,0.326,0.321
machine learning engineer,0.024,0.025,0.021,0.026
researcher,0.012,0.011,0.016,0.011
senior data analyst,0.008,0.008,0.011,0.005


## 4. Data leakage checks

Two kinds, checked explicitly rather than assumed:

1. **Row-level** — the same candidate appearing in more than one split.
2. **Feature-level** — `screening_score` accidentally showing up as a
   standalone number inside `input_text`. A raw substring search (`str(score)
   in text`) produces false positives constantly — a score of 25 matches
   inside "2025", a GPA of "3.25", a phone/zip fragment — none of which is
   real leakage. This checks for the score as a genuine standalone token
   (word boundaries) instead, and prints any flagged rows before asserting
   so a real leak is distinguishable from a coincidence.

In [9]:
train_ids = set(train_df["id"])
val_ids = set(val_df["id"])
test_ids = set(test_df["id"])

assert not (train_ids & val_ids), "Candidate overlap between train and val!"
assert not (train_ids & test_ids), "Candidate overlap between train and test!"
assert not (val_ids & test_ids), "Candidate overlap between val and test!"


def score_leaked(score, text: str) -> bool:
    try:
        score_str = str(int(round(float(score))))
    except (TypeError, ValueError):
        return False
    pattern = r"\b" + re.escape(score_str) + r"\b"
    return bool(re.search(pattern, str(text)))


for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    leaked_mask = split_df.apply(lambda row: score_leaked(row["screening_score"], row["input_text"]), axis=1)
    if leaked_mask.any():
        print(f"--- {split_name}: {leaked_mask.sum()} row(s) flagged -- inspect before trusting the assert ---")
        print(split_df.loc[leaked_mask, ["screening_score", "input_text"]].head(10))
    assert not leaked_mask.any(), f"screening_score appears as a standalone number inside input_text in {split_name}!"

print("No candidate overlap across splits, and no screening_score leakage into input text.")

No candidate overlap across splits, and no screening_score leakage into input text.


## 5. Tokenizer — with the pad-token fix Qwen needs

BERT's tokenizer always ships with a pad token; Qwen2.5's does not,
since causal LMs are normally used for generation rather than batched
classification/regression. Without this, batches with more than one row
fail outright.

In [10]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"pad_token: {tokenizer.pad_token!r}  (id: {tokenizer.pad_token_id})")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pad_token: '<|endoftext|>'  (id: 151643)


In [11]:
def to_hf_dataset(split_df: pd.DataFrame) -> Dataset:
    out = split_df[["input_text", "screening_score"]].rename(columns={"screening_score": "labels"})
    out["labels"] = out["labels"].astype(float)
    return Dataset.from_pandas(out.reset_index(drop=True))


def tokenize_fn(batch):
    return tokenizer(batch["input_text"], truncation=True, max_length=MAX_LENGTH)


train_ds = to_hf_dataset(train_df).map(tokenize_fn, batched=True, remove_columns=["input_text"])
val_ds = to_hf_dataset(val_df).map(tokenize_fn, batched=True, remove_columns=["input_text"])
test_ds = to_hf_dataset(test_df).map(tokenize_fn, batched=True, remove_columns=["input_text"])

for ds in (train_ds, val_ds, test_ds):
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(train_ds)

Map:   0%|          | 0/886 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 886
})


## 6. QLoRA setup — 4-bit (NF4) base model + LoRA adapters

**`BitsAndBytesConfig`** is what makes this QLoRA rather than plain LoRA:

- `load_in_4bit=True` — the base model's weights are compressed to 4 bits
  and stay frozen at that precision the entire time. Only the LoRA
  adapter matrices and the regression head are ever computed at higher
  precision.
- `bnb_4bit_quant_type="nf4"` — NF4 (NormalFloat4) is a quantization
  scheme designed specifically for neural network weight distributions
  (which tend to be roughly normal/bell-shaped) — it represents that
  distribution more accurately than a plain uniform 4-bit scheme at the
  same bit width.
- `bnb_4bit_use_double_quant=True` — quantizes the quantization constants
  themselves for a further small memory saving, essentially free in
  terms of accuracy cost.
- `bnb_4bit_compute_dtype=torch.float16` — the dtype used for the actual
  matrix-multiply math during forward/backward, even though weights are
  stored in 4-bit. This matches `TrainingArguments(fp16=True)` below —
  keeping compute dtype and training precision aligned is what avoids
  the `RuntimeError: Found dtype Float but expected BFloat16` class of
  error from mismatched precision.

**`AutoModelForSequenceClassification(num_labels=1, problem_type="regression")`**
swaps Qwen's usual next-token-prediction head for a single-output
regression head. `model.config.pad_token_id` is set explicitly — the
model's own config needs this, not just the tokenizer's.

**`prepare_model_for_kbit_training()`** is required before attaching LoRA
on top of a 4-bit model — it handles numerical-stability setup (casting
norm layers to fp32, enabling gradient checkpointing hooks) that
quantized models need for gradients to flow correctly into the adapter.

**`LoraConfig`** — `target_modules` are Qwen2's real attention/MLP
projection names (verified against Qwen2.5's architecture, not BERT's
`query`/`value`). `modules_to_save=["score"]` keeps the brand-new
regression head fully trainable — it's randomly initialized (see the
`score.weight | MISSING` load report Qwen2 prints), so a low-rank LoRA
nudge alone wouldn't be enough; it needs full updates, same reasoning as
BERT's `classifier` head, just under Qwen2's actual name for it.

In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression",
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.pad_token_id = tokenizer.pad_token_id

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-3B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 29,935,616 || all params: 3,115,876,352 || trainable%: 0.9607


## 7. Metrics and loss — MAE, directly

A custom `Trainer` subclass makes MAE (L1 loss) the actual training
objective, not just something measured afterward — the default for
`problem_type="regression"` is MSE, which penalizes large errors more
than proportionally; using L1 loss instead means training loss and your
evaluation metric are literally the same quantity, directly comparable
across epochs.

In [13]:
class MAETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        preds = outputs.logits.squeeze(-1)
        loss = nn.functional.l1_loss(preds, labels.float())
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.asarray(predictions).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    mae = mean_absolute_error(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    return {"mae": mae, "rmse": rmse}

## 8. Training arguments — every field explained

| Argument | What it does |
|---|---|
| `output_dir` | Where checkpoints and logs get written during training. |
| `num_train_epochs` | Full passes over the training set. More epochs = more chances to refine, but also more risk of overfitting past the point where validation MAE stops improving — this is exactly why `load_best_model_at_end` (below) matters. |
| `per_device_train_batch_size` | How many examples get processed together in one forward/backward pass. Bounded by GPU memory — lower this first if you hit an out-of-memory error. |
| `per_device_eval_batch_size` | Same idea, for evaluation. Can safely be larger than the training batch size since no gradients/optimizer state need to be held during eval. |
| `gradient_accumulation_steps` | Accumulates gradients across this many batches before an actual weight update — simulates a larger effective batch size (`train_batch_size x this`) without needing the memory to hold that many examples at once. |
| `learning_rate` | How large each gradient step is. LoRA/QLoRA adapters are small and randomly initialized, so they tolerate (and benefit from) a higher learning rate than full-model fine-tuning would use. |
| `eval_strategy="epoch"` | Run evaluation on `val_ds` at the end of every epoch (older `transformers` versions call this `evaluation_strategy`). |
| `save_strategy="epoch"` | Save a checkpoint at the end of every epoch, matching the eval schedule — needed for `load_best_model_at_end` to have checkpoints to choose between. |
| `load_best_model_at_end=True` | After training finishes, reload whichever epoch's checkpoint had the best `metric_for_best_model` value — not necessarily the last epoch. |
| `metric_for_best_model="mae"` | Which of `compute_metrics`'s outputs decides "best." |
| `greater_is_better=False` | **Easy to get backwards.** For MAE, lower is better — this flag has to be set explicitly, since `TrainingArguments` defaults to assuming higher is better (fine for accuracy, wrong for an error metric). |
| `fp16=torch.cuda.is_available()` | Enables mixed-precision training on GPU. Matched against `bnb_4bit_compute_dtype=torch.float16` above — mismatched precision settings between the quantization config and the trainer is what produces dtype-mismatch errors during backprop. |
| `gradient_checkpointing=True` | Trades compute for memory — recomputes some intermediate activations during the backward pass instead of storing all of them, reducing peak memory at the cost of somewhat slower training. Often necessary to fit a 3B model's training on a single T4/P100. |
| `logging_steps` | How often (in training steps) to log the current loss. |
| `save_total_limit` | Caps how many checkpoints are kept on disk at once (older ones get deleted) — without this, epoch checkpoints for a 3B model can fill up disk quickly. |
| `report_to="none"` | Disables logging integrations (Weights & Biases, etc.) that aren't set up here. |
| `seed` | Fixes randomness in data shuffling and any stochastic training behavior, for reproducibility. |

In [14]:
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/qwen-qlora-screening-score",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="mae",
    greater_is_better=False,
    fp16=False,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    logging_steps=10,
    report_to="none",
    warmup_ratio=0.05,
    seed=SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = MAETrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,          # test_ds is deliberately NOT passed here
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [15]:
# Check if any training loss was ever logged as NaN before this eval
for entry in trainer.state.log_history:
    if "loss" in entry and (entry["loss"] != entry["loss"]):  # NaN != NaN is True
        print("NaN loss at step", entry.get("step"), entry)
        break

# Check if the model's weights themselves are already NaN
for name, param in model.named_parameters():
    if param.requires_grad and torch.isnan(param).any():
        print(f"NaN found in parameter: {name}")

## 9. Train

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Rmse
1,85.810742,21.971710,21.971710,28.017740
2,95.954419,23.680737,23.680737,26.497189
3,87.207056,20.727814,20.727818,28.804325
4,83.143604,20.006413,20.006409,26.118014
5,75.508130,18.674271,18.674271,25.580957
6,65.788586,18.140394,18.140394,26.215107
7,70.090820,17.855869,17.855869,25.869127
8,63.353149,16.499916,16.499916,25.087188
9,62.503857,17.021206,17.021208,26.119222
10,50.224246,17.154556,17.154558,26.190652


TrainOutput(global_step=560, training_loss=76.67729993547712, metrics={'train_runtime': 4769.4449, 'train_samples_per_second': 1.858, 'train_steps_per_second': 0.117, 'total_flos': 6158473156362240.0, 'train_loss': 76.67729993547712, 'epoch': 10.0})

In [17]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)   # should be ~16.4999

/kaggle/working/qwen-qlora-screening-score/checkpoint-448
16.499916076660156


In [18]:
train_metrics = trainer.evaluate(train_ds)
print(train_metrics)

{'eval_loss': 13.971685409545898, 'eval_mae': 13.971685409545898, 'eval_rmse': 22.352039000076424, 'eval_runtime': 128.9259, 'eval_samples_per_second': 6.872, 'eval_steps_per_second': 0.861, 'epoch': 10.0}


In [19]:
naive_pred = train_df["screening_score"].mean()
naive_mae = mean_absolute_error(val_df["screening_score"], [naive_pred] * len(val_df))
print(f"Naive baseline (predict mean) MAE: {naive_mae:.2f}")
print(f"Fine-tuned model val MAE:          20.39")
print(f"Fine-tuned model train MAE:        19.80")

Naive baseline (predict mean) MAE: 29.15
Fine-tuned model val MAE:          20.39
Fine-tuned model train MAE:        19.80


## 10. Validation results

In [20]:
val_metrics = trainer.evaluate(val_ds)
print(f"Validation MAE:  {val_metrics['eval_mae']:.3f}")
print(f"Validation RMSE: {val_metrics['eval_rmse']:.3f}")

Validation MAE:  16.500
Validation RMSE: 25.087


## 11. Final test evaluation — run this once

`test_ds` has not been touched anywhere above — not for model selection,
not for hyperparameters, not for early stopping. That's what makes this
number trustworthy. If you go back and change settings based on what you
see here, the test set stops being genuinely held-out.

In [21]:
test_output = trainer.predict(test_ds)
test_preds = np.asarray(test_output.predictions).reshape(-1)
test_labels = np.asarray(test_output.label_ids).reshape(-1)

test_mae = mean_absolute_error(test_labels, test_preds)
test_rmse = np.sqrt(mean_squared_error(test_labels, test_preds))
naive_mae = np.mean(np.abs(train_df["screening_score"].mean() - test_labels))

print(f"Test MAE:  {test_mae:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")
print(f"Naive (predict-the-mean) MAE: {naive_mae:.3f}")
print(f"Spearman (rank agreement): {spearmanr(test_preds, test_labels).correlation:.3f}")

Test MAE:  16.570
Test RMSE: 25.296
Naive (predict-the-mean) MAE: 30.252
Spearman (rank agreement): 0.548


In [22]:
test_preds = test_output.predictions.reshape(-1)
test_df["pred"] = test_preds
test_df["abs_err"] = (test_df["pred"] - test_df["screening_score"]).abs()

print(test_df["abs_err"].describe(percentiles=[.5, .75, .9, .95]))
print(test_df.nlargest(15, "abs_err")[["primary_title", "screening_score", "pred", "abs_err", "input_text"]])
print(test_df.groupby("primary_title")["abs_err"].agg(["mean", "count"]).sort_values("mean", ascending=False).head(15))
print(train_df["screening_score"].describe())

count    190.000000
mean      16.569757
std       19.163687
min        0.083984
50%        6.875000
75%       22.875000
90%       49.550000
95%       54.275000
max       65.000000
Name: abs_err, dtype: float64
                  primary_title  screening_score    pred  abs_err  \
137             data strategist               20  85.000   65.000   
45    machine learning engineer               95  30.375   64.625   
98               data scientist               95  31.875   63.125   
122                data analyst               20  79.500   59.500   
36               data scientist               90  31.875   58.125   
46                  ml engineer               25  82.500   57.500   
182              data scientist               25  82.000   57.000   
141              data scientist               90  33.750   56.250   
175                data analyst               30  85.500   55.500   
171                data analyst               30  84.500   54.500   
178  project operations manager

In [23]:
STUDENT_PATTERN = re.compile(
    r"\b(student|students|intern|interns|internship|internships|undergrad\w*|"
    r"fresher|trainee|co-?op|entry[- ]level|recent grad\w*|graduate student)\b", re.I)

tagged = df["experience"].astype(str).str.lower().eq("student/intern-level")
says_student = df["raw_text"].fillna("").str.contains(STUDENT_PATTERN)

print(df.groupby([tagged.rename("tagged_student"), says_student.rename("text_says_student")])
        ["screening_score"].agg(["mean", "std", "count"]))

                                       mean        std  count
tagged_student text_says_student                             
False          False              56.286678  32.969306    593
               True               73.181818  33.784074     11
True           False              66.872951  30.722181    488
               True               71.155172  29.586242    174


/tmp/ipykernel_22/1760291385.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  says_student = df["raw_text"].fillna("").str.contains(STUDENT_PATTERN)


In [24]:
sus = tagged & ~says_student
print(df.assign(sus=sus).groupby("sus")[["education", "other_info", "location"]]
        .agg(lambda s: s.notna().mean()).round(2))

# Quick proxy: does the model's input do better with or without the tag?
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

def text_variant(d, blank_fixed):
    d = d.copy()
    if blank_fixed:
        m = (d["experience"].astype(str).str.lower().eq("student/intern-level")
             & ~d["raw_text"].fillna("").str.contains(STUDENT_PATTERN))
        d.loc[m, "experience"] = None
    return d.apply(build_input_text, axis=1)

for blank in [False, True]:
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    Xtr = vec.fit_transform(text_variant(train_df, blank))
    Xva = vec.transform(text_variant(val_df, blank))
    p = Ridge(alpha=1.0).fit(Xtr, train_df["screening_score"]).predict(Xva)
    print(f"blank_fixed={blank}: val MAE={mean_absolute_error(val_df['screening_score'], p):.2f}, "
          f"Spearman={spearmanr(p, val_df['screening_score']).correlation:.3f}")

       education  other_info  location
sus                                   
False       0.34        0.44       1.0
True        0.59        0.62       1.0
blank_fixed=False: val MAE=22.91, Spearman=0.322
blank_fixed=True: val MAE=23.52, Spearman=0.315


/tmp/ipykernel_22/382167494.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & ~d["raw_text"].fillna("").str.contains(STUDENT_PATTERN))
/tmp/ipykernel_22/382167494.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & ~d["raw_text"].fillna("").str.contains(STUDENT_PATTERN))


In [25]:
title_counts = df["primary_title"].value_counts()
print(f"Distinct raw titles: {len(title_counts)}")
print(f"Titles with 20+ rows: {(title_counts >= 20).sum()}")
print(f"Titles with 5-19 rows: {((title_counts >= 5) & (title_counts < 20)).sum()}")
print(f"Titles with <5 rows: {(title_counts < 5).sum()}")
print(title_counts.head(20))

Distinct raw titles: 258
Titles with 20+ rows: 7
Titles with 5-19 rows: 11
Titles with <5 rows: 240
primary_title
data scientist                   409
data analyst                     271
Unknown                           75
software engineer                 47
data engineer                     43
machine learning engineer         31
student                           21
software developer                16
researcher                        15
business analyst                  11
senior data analyst               10
ai engineer                        7
computer science                   7
research assistant                 6
analyst                            6
aiml engineer                      6
ml engineer                        5
full stack developer               5
full-stack developer               4
business intelligence analyst      4
Name: count, dtype: int64


In [26]:
import pandas as pd

# ---- Row count per job title, in val and test separately ----
val_counts = val_df["primary_title"].value_counts().rename("val_n")
test_counts = test_df["primary_title"].value_counts().rename("test_n")

split_counts = pd.concat([val_counts, test_counts], axis=1).fillna(0).astype(int)
split_counts["val_plus_test_n"] = split_counts["val_n"] + split_counts["test_n"]

# ---- Also show the full (train+val+test) count for context ----
full_counts = df["primary_title"].value_counts().rename("full_n")
split_counts = split_counts.join(full_counts)

# Sort by total dataset size, most common titles first
split_counts = split_counts.sort_values("full_n", ascending=False)

print(f"Total distinct titles: {len(split_counts)}")
print(split_counts.head(30))

Total distinct titles: 91
                                  val_n  test_n  val_plus_test_n  full_n
primary_title                                                           
data scientist                       62      61              123     409
data analyst                         40      41               81     271
Unknown                              12      11               23      75
software engineer                     7       7               14      47
data engineer                         6       7               13      43
machine learning engineer             4       5                9      31
student                               3       3                6      21
software developer                    2       3                5      16
researcher                            3       2                5      15
business analyst                      1       2                3      11
senior data analyst                   2       1                3      10
computer science         

In [27]:
df[df["primary_title"] == "Unknown"][["job_title", "raw_text"]].head(20)

,job_title,raw_text
5,None,student at kennesaw state university
20,None,detailed oriented with excellent communication...
89,None,alumni of california state university channel ...
95,None,data-driven decision-making enthusiast expert ...
106,None,itil 4 comptia a
172,None,.
180,None,multi-competent generalist
187,None,former graduate student at university of north...
227,None,motivated dedicated and hard working
234,None,works at unique computing llc


In [28]:
unknown_rows = df[df["primary_title"] == "Unknown"]
print(unknown_rows[["raw_text", "job_title", "education", "other_info"]].sample(15, random_state=42))

                                               raw_text job_title  \
106                                    itil 4 comptia a      None   
1072               motivated dedicated and hard working      None   
250   an outgoing quick learner and determined probl...      None   
5                  student at kennesaw state university      None   
697                                     reefpoint group      None   
1053  assistant professor of electrical and computer...      None   
587                                ms in bioinformatics      None   
285                               mathematical modeling      None   
1221  B.Sc.Hons in Statistics Operations Research Fa...      None   
1119  assistant professor in mathematics numerical a...      None   
685                         student at edgewood college      None   
805                             university of cambridge      None   
866                   graduate at iowa state university      None   
482                             da

In [29]:
usable_for_spearman = split_counts[split_counts["val_plus_test_n"] >= 5]
print(f"Titles with 5+ combined val/test rows: {len(usable_for_spearman)}")
print(usable_for_spearman)

Titles with 5+ combined val/test rows: 10
                           val_n  test_n  val_plus_test_n  full_n
primary_title                                                    
data scientist                62      61              123     409
data analyst                  40      41               81     271
Unknown                       12      11               23      75
software engineer              7       7               14      47
data engineer                  6       7               13      43
machine learning engineer      4       5                9      31
student                        3       3                6      21
software developer             2       3                5      16
researcher                     3       2                5      15
ai engineer                    3       2                5       7


## 12. Save the LoRA adapter

Only the small adapter weights (a few MB) get saved — not a copy of the
quantized base model.

In [30]:
ADAPTER_DIR = f"{OUTPUT_DIR}/qwen-qlora-screening-score-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved adapter to {ADAPTER_DIR}")

Saved adapter to /kaggle/working/qwen-qlora-screening-score-adapter


## 13. Post-fine-tuning diagnostics

Run these **after** Section 11 (they use `test_output`, `trainer`, and the
split dataframes). Nothing here retrains the model or touches the saved
adapter, so it is safe to run before or after Section 12.

1. **Residual analysis** - where the test errors come from (worst rows, error by title).
2. **Label shape and label noise** - is `screening_score` two-tiered, and do identical inputs get different labels?
3. **`experience` tag audit** - how often "student/intern-level" is tagged without student wording, and whether that tag carries signal.
4. **Cheap ablation** - TF-IDF + Ridge with and without the suspect tag, as a directional hint before spending a full retraining run.

### 13.1 Residual analysis on the test set

In [31]:

from IPython.display import display

# Reuse test_output from Section 11; recompute only if it is missing.
if "test_output" not in globals():
    test_output = trainer.predict(test_ds)

test_preds = np.asarray(test_output.predictions).reshape(-1)

analysis_df = test_df.copy().reset_index(drop=True)
analysis_df["pred"] = test_preds
analysis_df["abs_err"] = (analysis_df["pred"] - analysis_df["screening_score"]).abs()

print("Absolute error distribution (test):")
print(analysis_df["abs_err"].describe(percentiles=[.5, .75, .9, .95]))

share_top10 = analysis_df.nlargest(int(0.1 * len(analysis_df)), "abs_err")["abs_err"].sum() / analysis_df["abs_err"].sum()
print(f"\nShare of total error coming from the worst 10% of rows: {share_top10:.1%}")

pd.set_option("display.max_colwidth", 110)
print("\n15 largest errors:")
display(analysis_df.nlargest(15, "abs_err")[["primary_title", "screening_score", "pred", "abs_err", "input_text"]])

# Only titles with 3+ test rows - single-row titles are pure noise.
by_title = analysis_df.groupby("primary_title")["abs_err"].agg(["mean", "count"])
print("\nError by title (titles with 3+ test rows):")
display(by_title[by_title["count"] >= 3].sort_values("mean", ascending=False))








Absolute error distribution (test):
count    190.000000
mean      16.569757
std       19.163687
min        0.083984
50%        6.875000
75%       22.875000
90%       49.550000
95%       54.275000
max       65.000000
Name: abs_err, dtype: float64

Share of total error coming from the worst 10% of rows: 33.5%

15 largest errors:


,primary_title,screening_score,pred,abs_err,input_text
137,data strategist,20,85.000,65.000,Job Title: data strategist | Experience: student/intern-level | Education: morehouse school of medicine ms...
45,machine learning engineer,95,30.375,64.625,Job Title: machine learning engineer | Location: United States
98,data scientist,95,31.875,63.125,Job Title: data scientist | Location: United States
122,data analyst,20,79.500,59.500,Job Title: data analyst | Experience: student/intern-level | Location: United States
36,data scientist,90,31.875,58.125,Job Title: data scientist | Location: United States
46,ml engineer,25,82.500,57.500,"Job Title: ml engineer | Other Info: python, llms, nlp, ai, financial services skills | Location: United S..."
182,data scientist,25,82.000,57.000,"Job Title: data scientist | Experience: student/intern-level | Education: mscs at asu | Other Info: ai, ad..."
141,data scientist,90,33.750,56.250,"Job Title: data scientist | Other Info: machine learning, llms skills | Location: United States"
175,data analyst,30,85.500,55.500,Job Title: data analyst | Experience: student/intern-level | Education: ms in business intelligence analyt...
171,data analyst,30,84.500,54.500,"Job Title: data analyst | Experience: student/intern-level | Education: ms in analytics, usc viterbi, uc s..."



Error by title (titles with 3+ test rows):


,mean,count
primary_title,,
student,23.500000,3
data scientist,19.226538,61
Unknown,15.622869,11
machine learning engineer,14.701758,5
data analyst,13.009289,41
software developer,10.166667,3
data engineer,7.357143,7
software engineer,4.357143,7


### 13.2 Label shape and label noise
A two-humped label distribution explains why L1 loss makes the model commit
to one tier and produces large wrong-tier misses. Identical `input_text`
with different labels is a noise floor no model can beat.

In [32]:
print("Train label distribution:")
print(pd.cut(train_df["screening_score"], [-1, 20, 40, 60, 80, 100]).value_counts().sort_index())

print("\nPredicted-score distribution on test (are predictions collapsing into two tiers?):")
print(pd.cut(analysis_df["pred"], [-1000, 20, 40, 60, 80, 1000]).value_counts().sort_index())

dup = train_df.groupby("input_text")["screening_score"].agg(["count", "std", "min", "max"])
dup = dup[dup["count"] > 1].sort_values("std", ascending=False)
print(f"\n{len(dup)} distinct input_texts appear more than once in train. Highest label disagreement:")
display(dup.head(15))

Train label distribution:
screening_score
(-1, 20]      87
(20, 40]     251
(40, 60]       0
(60, 80]     203
(80, 100]    345
Name: count, dtype: int64

Predicted-score distribution on test (are predictions collapsing into two tiers?):
pred
(-1000, 20]    18
(20, 40]       33
(40, 60]        7
(60, 80]       48
(80, 1000]     84
Name: count, dtype: int64

20 distinct input_texts appear more than once in train. Highest label disagreement:


,count,std,min,max
input_text,,,,
"Job Title: data analyst | Experience: student/intern-level | Other Info: sql, python, data visualization skills | Location: United States",3,40.414519,25,95
Job Title: computational linguist | Location: United States,2,38.890873,25,80
Location: United States,19,28.087166,20,95
Job Title: software engineer | Location: United States,8,27.508116,20,90
Job Title: data analyst | Experience: senior-level | Location: United States,4,26.770631,25,80
Job Title: data scientist | Location: United States,24,26.229720,20,100
Job Title: data scientist | Experience: student/intern-level | Location: United States,6,25.797287,30,97
Job Title: data analyst | Location: United States,19,22.229531,30,100
Job Title: data analyst | Experience: student/intern-level | Location: United States,12,16.451029,32,100


### 13.3 `experience` tag audit

Compares rows the extraction tagged `student/intern-level` against whether
the raw text actually contains student wording. "graduate" and "alumni" are
excluded on purpose (a graduate can be many years out).

In [33]:
STUDENT_PATTERN = re.compile(
    r"\b(?:student|students|intern|interns|internship|internships|undergrad\w*|"
    r"fresher|trainee|co-?op|entry[- ]level|recent grad\w*|graduate student)\b",
    re.I,
)

tagged = df["experience"].astype(str).str.lower().eq("student/intern-level")
says_student = df["raw_text"].fillna("").str.contains(STUDENT_PATTERN)

print("Mean screening_score by (tagged as student, text actually says student):")
display(
    df.groupby([tagged.rename("tagged_student"), says_student.rename("text_says_student")])
      ["screening_score"].agg(["mean", "std", "count"])
)

suspect = tagged & ~says_student
print(f"\n{tagged.sum()} rows tagged student; {suspect.sum()} of them have no student wording in raw_text")
print("\nSample of suspect rows (read these before deciding to blank them):")
display(df.loc[suspect, ["raw_text", "experience", "screening_score"]].sample(min(15, int(suspect.sum())), random_state=42))

print("\nField fill rate, suspect rows vs everything else (is the tag a proxy for something else?):")
display(
    df.assign(suspect=suspect)
      .groupby("suspect")[["education", "other_info", "location"]]
      .agg(lambda s: s.notna().mean()).round(2)
)

Mean screening_score by (tagged as student, text actually says student):


mean        std  count
tagged_student text_says_student                             
False          False              56.286678  32.969306    593
               True               73.181818  33.784074     11
True           False              66.872951  30.722181    488
               True               71.155172  29.586242    174


662 rows tagged student; 488 of them have no student wording in raw_text

Sample of suspect rows (read these before deciding to blank them):


,raw_text,experience,screening_score
151,actively seeking full-time roles data analyst data engineer tableau certified power bi sql azure certified...,student/intern-level,80
590,data analyst texas state university alumni,student/intern-level,85
1107,m.s. in statistics from the university of illinois in urbana-champaign. actively seeking employment as a s...,student/intern-level,30
175,master of science in business analytics ex-clariant research analyst new business development data analyst...,student/intern-level,75
999,ms in data science uta actively looking for opportunities in spring 2025,student/intern-level,32
1229,B.Sc. Hons in Data Science Python AI ML LLM NLP Data Analysis,student/intern-level,0
481,data analyst masters in data science analyst modeling science data science machine learning enthusiast,student/intern-level,95
131,data analyst turning data into actionable insights skilled in sql python tableau power bi excel etl and da...,student/intern-level,80
191,b.s in computer science a passion for dance,student/intern-level,35
756,ms business analytics,student/intern-level,80



Field fill rate, suspect rows vs everything else (is the tag a proxy for something else?):


,education,other_info,location
suspect,,,
False,0.34,0.44,1.0
True,0.59,0.62,1.0


### 13.4 Cheap ablation - does blanking the suspect tag help?

TF-IDF + Ridge trains in seconds, so it gives a directional answer before
committing to another Qwen run. It is **not** the same model: treat a
difference under about 1 MAE on ~190 rows as noise, and confirm any real
change by retraining Qwen on the patched column with the same `SEED`.

In [34]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge


def text_variant(d: pd.DataFrame, blank_suspect: bool) -> pd.Series:
    d = d.copy()
    if blank_suspect:
        m = (d["experience"].astype(str).str.lower().eq("student/intern-level")
             & ~d["raw_text"].fillna("").str.contains(STUDENT_PATTERN))
        d.loc[m, "experience"] = None
    return d.apply(build_input_text, axis=1)


for blank in [False, True]:
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    Xtr = vec.fit_transform(text_variant(train_df, blank))
    Xva = vec.transform(text_variant(val_df, blank))
    model_r = Ridge(alpha=1.0).fit(Xtr, train_df["screening_score"])
    p = model_r.predict(Xva)
    mae = mean_absolute_error(val_df["screening_score"], p)
    rho = spearmanr(p, val_df["screening_score"]).correlation
    print(f"blank_suspect_tag={blank!s:5}  val MAE={mae:.2f}  Spearman={rho:.3f}")

blank_suspect_tag=False  val MAE=22.91  Spearman=0.322
blank_suspect_tag=True   val MAE=23.52  Spearman=0.315


## 14. Feature importance — no retraining needed

Runs on the model already in memory after Section 9's `trainer.train()`.
Uses `val_df` (never `test_df`, kept untouched as your final held-out
check). Two methods:

- **14.1 Leave-one-out** — blank each field, see how much validation MAE
  gets worse.
- **14.2 Permutation importance** — shuffle each field's values across
  rows instead of removing it. Generally the more trustworthy of the two,
  since the field stays present (so the model can't just key off
  "this field is missing" as its own signal) and only the field's link
  to the true label gets broken.

Given the single-candidate ablation earlier showed a 33-point swing from
`location` alone, expect it to dominate both tables below across the
full validation set.

In [35]:
@torch.no_grad()
def predict_batch(texts, batch_size=32):
    model.eval()
    preds = []
    for start in range(0, len(texts), batch_size):
        chunk = texts[start:start + batch_size]
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True,
                            max_length=MAX_LENGTH, padding=True).to(model.device)
        outputs = model(**inputs)
        preds.extend(outputs.logits.squeeze(-1).float().cpu().tolist())
    return np.array(preds)


baseline_preds = predict_batch(val_df["input_text"].tolist())
baseline_mae = mean_absolute_error(val_df["screening_score"], baseline_preds)
print(f"Baseline val MAE: {baseline_mae:.2f}")

FEATURE_FIELDS = ["job_title", "experience", "education", "other_info", "location"]

Baseline val MAE: 16.50


In [36]:
loo_results = {}
for field in FEATURE_FIELDS:
    modified = val_df.copy()
    modified[field] = None
    texts = modified.apply(build_input_text, axis=1)
    preds = predict_batch(texts.tolist())
    mae = mean_absolute_error(val_df["screening_score"], preds)
    loo_results[field] = mae - baseline_mae

loo_df = (
    pd.DataFrame.from_dict(loo_results, orient="index", columns=["mae_increase_when_removed"])
      .sort_values("mae_increase_when_removed", ascending=False)
)
print("Positive = model relies on this field (removing it hurts MAE). Near zero = field barely used.\n")
display(loo_df)

Positive = model relies on this field (removing it hurts MAE). Near zero = field barely used.



,mae_increase_when_removed
location,8.594903
job_title,2.580458
experience,1.845170
education,0.667105
other_info,0.575375


In [37]:
rng = np.random.default_rng(SEED)

perm_results = {}
for field in FEATURE_FIELDS:
    shuffled = val_df.copy()
    shuffled[field] = rng.permutation(shuffled[field].values)
    texts = shuffled.apply(build_input_text, axis=1)
    preds = predict_batch(texts.tolist())
    mae = mean_absolute_error(val_df["screening_score"], preds)
    perm_results[field] = mae - baseline_mae

perm_df = (
    pd.DataFrame.from_dict(perm_results, orient="index", columns=["mae_increase_when_shuffled"])
      .sort_values("mae_increase_when_shuffled", ascending=False)
)
print("Field kept present but its link to the true label is broken -- "
      "generally the more trustworthy of the two tables.\n")
display(perm_df)

Field kept present but its link to the true label is broken -- generally the more trustworthy of the two tables.



,mae_increase_when_shuffled
location,7.319058
job_title,5.264906
experience,1.059685
education,0.800068
other_info,0.714257


In [38]:
loc_means = (
    train_df.assign(location_norm=train_df["location"].str.lower())
            .groupby("location_norm")["screening_score"]
            .agg(["mean", "count"])
            .sort_values("count", ascending=False)
)
display(loc_means.head(10))

,mean,count
location_norm,,
united states,68.443152,774
india,0.000000,16
united kingdom,81.363636,11
pakistan,0.000000,9
united arab emirates,84.444444,9
brazil,41.875000,8
egypt,0.000000,8
nigeria,0.000000,7
bangladesh,0.000000,3
